In [ ]:
# 7-1.py
import RPi.GPIO as GPIO
import time
import matplotlib.pyplot as plt

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)

Led_pins = [21, 20, 16, 12, 7, 8, 25, 24]
DAC_pins = [26, 19, 13, 6, 5, 11, 9, 10]  
Comp_pin = 4
troyka_pin = 17

GPIO.setup(Led_pins, GPIO.OUT, initial = GPIO.LOW)
GPIO.setup(DAC_pins, GPIO.OUT, initial = GPIO.LOW)
GPIO.setup(Comp_pin, GPIO.IN)
GPIO.setup(troyka_pin, GPIO.OUT, initial = GPIO.LOW)

def decimal_to_binary(value):
    
    return [(value >> (7 - i)) & 1 for i in range(8)]

def adc():
    
    value = 0
    for bit in range(7, -1, -1):
        new_value = value | (1 << bit)
        GPIO.output(DAC_pins, decimal_to_binary(new_value))
        time.sleep(0.01)

        if GPIO.input(Comp_pin) == GPIO.HIGH:
            value = new_value
    return value

def measure_voltage():
    
    return adc() * 3.3 / 255

def update_leds(value):
    
    leds_on = min(8, round(value / 32))
    states = [GPIO.HIGH] * leds_on + [GPIO.LOW] * (8 - leds_on)
    GPIO.output(Led_pins, states)

def save_data(filename, data):
    
    with open(filename, 'w') as f:
        for value in data:
            f.write(f"{value}\n")

try:
    measurements = []
    start_time = time.time()

    GPIO.output(troyka_pin, GPIO.HIGH)
    print("Начало зарядки...")

    while True:
        voltage = measure_voltage()
        measurements.append(voltage)
        update_leds(int(voltage * 255 / 3.3))

        if voltage >= 3.2:  
            print("Зарядка завершена")
            break

    GPIO.output(troyka_pin, GPIO.LOW)
    print("Начало разрядки...")

    while True:
        voltage = measure_voltage()
        measurements.append(voltage)
        update_leds(int(voltage * 255 / 3.3))

        if voltage <= 0.1:
            print("Разрядка завершена")
            break

    duration = time.time() - start_time
    sample_count = len(measurements)

    save_data('data.txt', measurements)

    with open('settings.txt', 'w') as f:
        f.write(f"Частота дискретизации: {sample_count / duration:.2f} Гц\n")
        f.write(f"Шаг квантования: {3.3 / 255:.4f} В\n")

    print(f"Общее время: {duration:.2f} с")
    print(f"Количество измерений: {sample_count}")
    print(f"Частота дискретизации: {sample_count/duration:.2f} Гц")
    print(f"Шаг квантования: {3.3 / 255:.4f} В")

    plt.plot(measurements)
    plt.title("Процесс заряда/разряда конденсатора")
    plt.xlabel("Номер измерения")
    plt.ylabel("Напряжение (В)")
    plt.grid()
    plt.show()

except KeyboardInterrupt:
    print("\nИзмерение прервано пользователем")
except Exception as e:
    print(f"Ошибка: {str(e)}")
finally:
    GPIO.output(Led_pins, GPIO.LOW)
    GPIO.output(DAC_pins, GPIO.LOW)
    GPIO.output(troyka_pin, GPIO.LOW)
    GPIO.cleanup()
    print("Ресурсы освобождены")